# Copernicus Marine: Sea Surface Temperature & Chlorophyll — Mediterranean Sea
* Need to have an account on Copernicus Marine
* The ~/.netrc file contains the account credentials: your login and password for [Copernicus Marine](https://marine.copernicus.eu/)
```
$ more ~/.netrc
machine auth.marine.copernicus.eu
  login xxxxxxxxxx 
  password xxxxxxxxxxxxx
```
* Uses the **Mediterranean Sea Physics Reanalysis** (`MEDSEA_MULTIYEAR_PHY_006_004`) for SST
* Uses the **Mediterranean Sea Biogeochemistry Reanalysis** (`MEDSEA_MULTIYEAR_BGC_006_008`) for chlorophyll
* Monthly mean fields for August 2020

In [ ]:
import copernicusmarine
import hvplot.xarray
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
# Mediterranean bounding box
MED_BBOX = dict(
    minimum_longitude=-6,
    maximum_longitude=42,
    minimum_latitude=30,
    maximum_latitude=47,
)
YEAR_MONTH = "2020-08"

## Sea Surface Temperature
Dataset: `cmems_mod_med_phy-temp_my_4.2km_P1M-m` (monthly means, Mediterranean Physics Reanalysis)  
Variable: `thetao` — potential temperature (°C) at the surface (depth = 0)

In [ ]:
%%time
ds_sst = copernicusmarine.open_dataset(
    dataset_id="cmems_mod_med_phy-temp_my_4.2km_P1M-m",
    **MED_BBOX,
)
ds_sst

In [ ]:
%%time
# Select surface (nearest to 0 m) and the target month
sst = (
    ds_sst["thetao"]
    .sel(depth=0, method="nearest")
    .sel(time=YEAR_MONTH, method="nearest")
    .load()
)
print(f"SST range: {float(sst.min()):.1f} – {float(sst.max()):.1f} °C")
sst

In [ ]:
sst.hvplot(
    x="longitude",
    y="latitude",
    rasterize=True,
    geo=True,
    cmap="turbo",
    clabel="SST (°C)",
    tiles="OSM",
    frame_width=800,
    frame_height=450,
    title=f"Mediterranean Sea Surface Temperature — {YEAR_MONTH}",
    fontscale=1.2,
)

## Chlorophyll
Dataset: `cmems_mod_med_bgc-plankton_my_4.2km_P1M-m` (monthly means, Mediterranean Biogeochemistry Reanalysis)  
Variable: `chl` — total chlorophyll concentration (mg m⁻³) at the surface (depth = 0)

In [ ]:
%%time
ds_chl = copernicusmarine.open_dataset(
    dataset_id="cmems_mod_med_bgc-plankton_my_4.2km_P1M-m",
    **MED_BBOX,
)
ds_chl

In [ ]:
%%time
chl = (
    ds_chl["chl"]
    .sel(depth=0, method="nearest")
    .sel(time=YEAR_MONTH, method="nearest")
    .load()
)
print(f"Chlorophyll range: {float(chl.min()):.4f} – {float(chl.max()):.4f} mg/m³")
chl

In [ ]:
import numpy as np

# Log scale brings out spatial structure in chlorophyll
chl_log = np.log10(chl.where(chl > 0))
chl_log.attrs["long_name"] = "log₁₀ Chlorophyll"
chl_log.attrs["units"] = "log₁₀(mg m⁻³)"

chl_log.hvplot(
    x="longitude",
    y="latitude",
    rasterize=True,
    geo=True,
    cmap="viridis",
    clabel="log₁₀ Chlorophyll (mg m⁻³)",
    tiles="OSM",
    frame_width=800,
    frame_height=450,
    title=f"Mediterranean Chlorophyll (log₁₀ scale) — {YEAR_MONTH}",
    fontscale=1.2,
)

## Side-by-side comparison

In [ ]:
plot_sst = sst.hvplot(
    x="longitude", y="latitude",
    rasterize=True, geo=True,
    cmap="turbo", clabel="SST (°C)",
    tiles="OSM",
    frame_width=560, frame_height=350,
    title=f"SST — {YEAR_MONTH}",
)

plot_chl = chl_log.hvplot(
    x="longitude", y="latitude",
    rasterize=True, geo=True,
    cmap="viridis", clabel="log₁₀ Chl (mg m⁻³)",
    tiles="OSM",
    frame_width=560, frame_height=350,
    title=f"Chlorophyll — {YEAR_MONTH}",
)

(plot_sst + plot_chl).cols(2)